# 01 — Data Exploration: MRI + CT Volumes

**Goal**: Load one patient's paired MRI and CT NIfTI files, understand their properties, visualize axial slices, plot intensity histograms, and see what normalization does to the data.

This notebook is the starting point for understanding your dataset **before** training any model. Always explore your data first!

---

### What is a NIfTI file?
NIfTI (`.nii.gz`) is a standard format for storing 3D medical images. Each file is essentially a 3D array (or 4D for time series) plus metadata like voxel spacing (how many mm per pixel).

### What is a voxel?
A **voxel** (volume element) is the 3D equivalent of a pixel. In a brain CT, each voxel typically represents a 1mm × 1mm × 1mm cube of tissue. The intensity value stored at each voxel is a **Hounsfield Unit (HU)** in CT, or an arbitrary MRI signal intensity.

---
**Before running**: Make sure you have downloaded the SynthRAD2023 dataset (see `data/README.md`).
Update the `PATIENT_DIR` path below to point to a real patient folder.

In [ ]:
# ─── Setup & Imports ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Add src/ to path so we can use our normalization functions
sys.path.insert(0, str(Path('..') / 'src'))
from utils import normalize_mri, normalize_ct

# ─── CONFIGURE THIS PATH ──────────────────────────────────────────────────────
# Point this to one of your downloaded patient folders, e.g.:
#   PATIENT_DIR = Path('../data/brain/1BA001')
PATIENT_DIR = Path('../data/brain/1BA001')  # <-- change this!

MRI_PATH = PATIENT_DIR / 'mr.nii.gz'   # or 'mri.nii.gz' depending on the dataset
CT_PATH  = PATIENT_DIR / 'ct.nii.gz'

print(f'MRI file exists: {MRI_PATH.exists()}')
print(f'CT  file exists: {CT_PATH.exists()}')

## 1. Load the volumes

We use **nibabel** to load NIfTI files. The `.get_fdata()` method returns a NumPy array.

The `.header` attribute contains metadata: voxel spacing (in mm), orientation, etc.

In [ ]:
# Load the NIfTI volumes
mri_img = nib.load(str(MRI_PATH))
ct_img  = nib.load(str(CT_PATH))

# get_fdata() returns a float64 array; specify float32 to save memory
mri_vol = mri_img.get_fdata(dtype=np.float32)
ct_vol  = ct_img.get_fdata(dtype=np.float32)

# ─── Volume shape ─────────────────────────────────────────────────────────────
print('=== MRI Volume ===')
print(f'  Shape:         {mri_vol.shape}   (X × Y × Z axial slices)')
print(f'  Dtype:         {mri_vol.dtype}')
print(f'  Min intensity: {mri_vol.min():.2f}')
print(f'  Max intensity: {mri_vol.max():.2f}')
print(f'  Mean:          {mri_vol.mean():.2f}')

print()
print('=== CT Volume ===')
print(f'  Shape:         {ct_vol.shape}   (X × Y × Z axial slices)')
print(f'  Dtype:         {ct_vol.dtype}')
print(f'  Min intensity: {ct_vol.min():.2f} HU')
print(f'  Max intensity: {ct_vol.max():.2f} HU')
print(f'  Mean:          {ct_vol.mean():.2f} HU')

## 2. Voxel spacing (image resolution in mm)

Voxel spacing tells you the physical size of each voxel. For example, `(1.0, 1.0, 2.5)` means each voxel is 1mm × 1mm × 2.5mm.

This is important because:
- It affects the **aspect ratio** of visualized slices.
- Anisotropic spacing (e.g. thick slices in Z) means neighbouring slices may look very different.
- For training, we resize all slices to 256×256, which ignores the original spacing — a simplification that works for our portfolio project.

In [ ]:
# Extract voxel spacing from the NIfTI header
mri_spacing = mri_img.header.get_zooms()  # Returns (dx, dy, dz) in mm
ct_spacing  = ct_img.header.get_zooms()

print(f'MRI voxel spacing: {mri_spacing[0]:.2f} x {mri_spacing[1]:.2f} x {mri_spacing[2]:.2f} mm')
print(f'CT  voxel spacing: {ct_spacing[0]:.2f} x {ct_spacing[1]:.2f} x {ct_spacing[2]:.2f} mm')

n_mri_slices = mri_vol.shape[2]
n_ct_slices  = ct_vol.shape[2]
print(f'\nNumber of axial slices: MRI={n_mri_slices}, CT={n_ct_slices}')

if mri_vol.shape != ct_vol.shape:
    print('\n⚠️  MRI and CT have different shapes — the dataset may need resampling.')
    print('   SynthRAD2023 volumes are pre-registered, so shapes should match.')
else:
    print('\n✓ MRI and CT shapes match — volumes are co-registered.')

## 3. Visualize 6 axial slices side by side

We'll look at 6 evenly-spaced slices along the Z-axis.

**Top row**: MRI slices (T1-weighted) — soft tissue contrast, bone appears dark.
**Bottom row**: CT slices — bone appears bright (white = dense = high HU), soft tissue is grey.

Notice how the two modalities show the **same anatomy** in completely different appearances. Our model's job is to learn this mapping!

In [ ]:
n_slices = mri_vol.shape[2]
# Pick 6 evenly-spaced slices, avoiding the very first and last (often empty)
slice_indices = np.linspace(int(n_slices * 0.1), int(n_slices * 0.9), 6, dtype=int)

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
fig.suptitle('Paired MRI (top) and CT (bottom) — 6 Axial Slices', fontsize=15, fontweight='bold')

for col, z_idx in enumerate(slice_indices):
    mri_slice = mri_vol[:, :, z_idx]
    ct_slice  = ct_vol[:, :, z_idx]

    # MRI: use percentile clipping for display (removes extreme bright spots)
    mri_vmin, mri_vmax = np.percentile(mri_slice, 1), np.percentile(mri_slice, 99)
    axes[0, col].imshow(mri_slice.T, cmap='gray', vmin=mri_vmin, vmax=mri_vmax, origin='lower')
    axes[0, col].set_title(f'z = {z_idx}', fontsize=10)
    axes[0, col].axis('off')

    # CT: clip to [-200, 1000] HU range for soft tissue + bone visibility
    axes[1, col].imshow(ct_slice.T, cmap='gray', vmin=-200, vmax=1000, origin='lower')
    axes[1, col].axis('off')

# Add row labels
axes[0, 0].set_ylabel('MRI', fontsize=13, rotation=90, labelpad=10)
axes[1, 0].set_ylabel('CT', fontsize=13, rotation=90, labelpad=10)

plt.tight_layout()
plt.savefig('../outputs/01_slice_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to outputs/01_slice_comparison.png')

## 4. Intensity Histograms

A **histogram** shows the distribution of intensity values across the entire volume.

Key observations:
- **MRI**: The distribution is modality- and scanner-dependent. Values are arbitrary (not standardized). A large spike near 0 represents the air background.
- **CT**: Values are in Hounsfield Units (HU), a physical scale:
  - Air: ~−1000 HU
  - Water / soft tissue: ~0–80 HU
  - Bone: ~400–1000 HU

Understanding these distributions motivates our normalization strategy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ─── MRI histogram ────────────────────────────────────────────────────────────
# Exclude near-zero voxels (background air) from the display
mri_nonzero = mri_vol[mri_vol > mri_vol.max() * 0.01].ravel()
axes[0].hist(mri_nonzero, bins=100, color='steelblue', alpha=0.8, edgecolor='none')
axes[0].set_title('MRI Intensity Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Signal Intensity (arbitrary units)', fontsize=11)
axes[0].set_ylabel('Voxel Count', fontsize=11)
axes[0].axvline(np.percentile(mri_nonzero, 1),  color='red',    linestyle='--', label='1st percentile')
axes[0].axvline(np.percentile(mri_nonzero, 99), color='orange', linestyle='--', label='99th percentile')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# ─── CT histogram ─────────────────────────────────────────────────────────────
ct_clipped = np.clip(ct_vol.ravel(), -1500, 1500)  # Clip for display
axes[1].hist(ct_clipped, bins=150, color='coral', alpha=0.8, edgecolor='none')
axes[1].set_title('CT Intensity Distribution (Hounsfield Units)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hounsfield Units (HU)', fontsize=11)
axes[1].set_ylabel('Voxel Count', fontsize=11)

# Annotate key HU ranges
for label, x_pos, color in [
    ('Air\n(~−1000)', -1000, 'navy'),
    ('Water / Tissue\n(~0–80)', 40, 'green'),
    ('Bone\n(~400+)', 700, 'brown'),
]:
    axes[1].axvline(x_pos, color=color, linestyle=':', alpha=0.7)
    axes[1].text(x_pos, axes[1].get_ylim()[1] * 0.9, label,
                 fontsize=8, color=color, ha='center', va='top')

axes[1].axvline(-1000, color='gray', linestyle='--', alpha=0.5, label='Clip bounds (±1000)')
axes[1].axvline(1000,  color='gray', linestyle='--', alpha=0.5)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/01_intensity_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Normalization — Before and After

Before feeding slices into the neural network, we normalize them:

| Modality | Input Range | Normalization | Output Range |
|----------|------------|---------------|-------------|
| MRI | Arbitrary | Percentile clip (1–99) → min-max | **[0, 1]** |
| CT  | Hounsfield Units | Clip to [−1000, +1000] → linear map | **[−1, 1]** |

Why [−1, 1] for CT? Because the U-Net uses a **tanh** output activation, which naturally outputs values in [−1, 1]. This makes the training targets consistent with the model's output range.

Why percentile clipping for MRI? MRI scanners produce inconsistent intensities across scanners and patients. Percentile clipping clips the 1% brightest and darkest pixels, removing scanner-specific noise/artifacts before scaling.

In [ ]:
# Pick a middle slice for demonstration
z = mri_vol.shape[2] // 2
mri_raw   = mri_vol[:, :, z]
ct_raw    = ct_vol[:, :, z]

# Apply our normalization functions
mri_norm  = normalize_mri(mri_raw)
ct_norm   = normalize_ct(ct_raw)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'Normalization Demo — Axial Slice z={z}', fontsize=14, fontweight='bold')

# ─── MRI: before and after ────────────────────────────────────────────────────
im0 = axes[0, 0].imshow(mri_raw.T, cmap='gray', origin='lower')
axes[0, 0].set_title(f'MRI — Raw\nrange: [{mri_raw.min():.1f}, {mri_raw.max():.1f}]', fontsize=11)
axes[0, 0].axis('off')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

im1 = axes[0, 1].imshow(mri_norm.T, cmap='gray', vmin=0, vmax=1, origin='lower')
axes[0, 1].set_title(f'MRI — Normalized\nrange: [{mri_norm.min():.3f}, {mri_norm.max():.3f}]', fontsize=11)
axes[0, 1].axis('off')
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

# ─── CT: before and after ─────────────────────────────────────────────────────
im2 = axes[1, 0].imshow(ct_raw.T, cmap='gray', vmin=-200, vmax=1000, origin='lower')
axes[1, 0].set_title(f'CT — Raw (HU)\nrange: [{ct_raw.min():.1f}, {ct_raw.max():.1f}] HU', fontsize=11)
axes[1, 0].axis('off')
plt.colorbar(im2, ax=axes[1, 0], fraction=0.046, label='HU')

im3 = axes[1, 1].imshow(ct_norm.T, cmap='gray', vmin=-1, vmax=1, origin='lower')
axes[1, 1].set_title(f'CT — Normalized\nrange: [{ct_norm.min():.3f}, {ct_norm.max():.3f}]', fontsize=11)
axes[1, 1].axis('off')
plt.colorbar(im3, ax=axes[1, 1], fraction=0.046)

plt.tight_layout()
plt.savefig('../outputs/01_normalization_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ All figures saved to outputs/')

## Summary & Key Takeaways

| Property | MRI | CT |
|----------|-----|----|
| Units | Arbitrary signal | Hounsfield Units |
| Air background | ~0 | ~−1000 HU |
| Bone appearance | Dark (low signal) | Bright (high HU) |
| Soft tissue | Variable | ~0–80 HU |
| Normalization target | [0, 1] | [−1, 1] |

### Next steps:
1. **Train the baseline model**: `python src/train_baseline.py --data_dir data/brain`
2. **Train pix2pix**: `python src/train_pix2pix.py --data_dir data/brain`
3. **Evaluate**: `python src/evaluate.py --checkpoint checkpoints/pix2pix_best.pth`
4. **Run the app**: `python app/app.py --checkpoint checkpoints/pix2pix_best.pth`